# Advanced Models
- Try Gradient Boosting (e.g., XGBoost or LightGBM). 
- Perform light hyperparameter tuning (depth, learning rate, n_estimators)

Choosing Models:
- https://www.geeksforgeeks.org/machine-learning/gradientboosting-vs-adaboost-vs-xgboost-vs-catboost-vs-lightgbm/ - XGBoost is the best


In [17]:
# Importing modules
import pandas as pd
import numpy as np

from xgboost import XGBRegressor
from scipy.stats import loguniform
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV


In [5]:
# Importing Data
root = "C:/Users/donutii/Desktop/IDX-Exchange-Internship-Data-Science-61"
data_location = f"{root}/IDX_Exchange/deliverables"

testing_set = pd.read_csv(f'{data_location}/CRMLS_202605_testing_set.csv')
training_set = pd.read_csv(f'{data_location}/CRMLS_202505_202604_training_set.csv') 


In [14]:
#splitting it (reused from 03_baseline_model.ipynb)
def GradientBoost(training, testing):

    # Splitting training set into X and Y
        # Y
    target = training['ClosePrice']
    target_testing = testing['ClosePrice']

        # X
    vars = training.drop(columns=['ClosePrice', 'CloseDate'])
    vars_testing = testing.drop(columns='ClosePrice')

    # Train a model
    model = XGBRegressor().fit(vars, target)

    # print results
    #r_sq = model.predict(target_testing)

    # print(f'{len(vars.columns)}')
    
    # Testing results on training
    #print(f'R2 value for Training: {r_sq}')
    
    # validating on testing set:
    r_sq = model.score(vars_testing, target_testing)

    print(f'R2 value on Test Set: {r_sq}')

In [ ]:
# Perform normal gradient boosting
GradientBoost(training_set, testing_set)

R2 value on Test Set: 0.910275417118815


# Hyperparameter Tuning:
- Adapted from https://inria.github.io/scikit-learn-mooc/python_scripts/ensemble_hyperparameters.html
- This time, using the HistGradientBoostingRegressor (which fixes the issue with slow models)

In [19]:
param_distributions = {
    "max_iter": [100, 300, 500, 600, 1000],
    "max_leaf_nodes": [5, 10, 20, 50, 75, 100],
    "learning_rate": loguniform(0.01, 1),
}
search_cv = RandomizedSearchCV(
    HistGradientBoostingRegressor(),
    param_distributions=param_distributions,
    scoring="neg_mean_absolute_error",
    n_iter=20,
    random_state=0
)
search_cv.fit(training_set.drop(columns=['ClosePrice', 'CloseDate']), training_set['ClosePrice'])

columns = [f"param_{name}" for name in param_distributions.keys()]
columns += ["mean_test_error", "std_test_error"]
cv_results = pd.DataFrame(search_cv.cv_results_)
cv_results["mean_test_error"] = -cv_results["mean_test_score"]
cv_results["std_test_error"] = cv_results["std_test_score"]
cv_results[columns].sort_values(by="mean_test_error")

,param_max_iter,param_max_leaf_nodes,param_learning_rate,mean_test_error,std_test_error
11,1000,50,0.110585,131910.820222,4587.777593
19,600,100,0.018107,132744.997493,4482.246067
8,300,50,0.088556,132973.252430,4341.439543
12,500,75,0.023587,134192.054764,4495.902706
14,100,75,0.171898,135700.166225,4626.931660
13,100,75,0.137046,135748.081623,4414.372664
6,100,50,0.197854,138305.052847,4010.267352
18,600,20,0.215543,138579.071737,4892.668618
0,100,50,0.125207,138993.544080,4239.082533
15,1000,10,0.059290,144371.153300,3750.453355


In [24]:
# using the best parameters to train the histogram gradient boosting model
model = HistGradientBoostingRegressor(max_iter=1000, max_leaf_nodes=50, learning_rate=0.111).fit(
                                    training_set.drop(columns=['ClosePrice', 'CloseDate']), 
                                    training_set['ClosePrice'])

r_sq = model.score(testing_set.drop(columns='ClosePrice'), testing_set['ClosePrice'])

print(f'R2 value on Test Set: {r_sq}')

R2 value on Test Set: 0.9198337699327794
